# Synthetic Experiment — Table 1

One experiment per template (`Geo`, `Poly`, `Else`), each independent:
- 30 noisy trajectories from a single template
- Split: 20 train / 5 val / 5 test
- Degree-3 Bézier | diagonal loading covariance | coarse-to-fine grid search over λ

**Workflow:**
1. Run **Setup** cell
2. Run **Generate** cell — generates fresh data (does NOT save)
3. Run **Experiments** cell — computes OLS and ridge results
4. Inspect results in **Table 1** cell
5. If satisfied, run **Save** cell to persist data to disk
6. On subsequent runs: run **Load** cell instead of Generate

In [1]:
%load_ext autoreload
%autoreload 2
#%env JAX_PLATFORM_NAME=gpu
#%env XLA_PYTHON_CLIENT_PREALLOCATE=false
import os
import numpy as np
import numpy.linalg as lg
import jax
from morphomatics.manifold import Sphere, PowerManifold
from morphomatics.stats import ExponentialBarycenter
from TimeSeries.stats import sph_correlated_trjs
from TimeSeries.main import pred
from TimeSeries.model import Reg, RidgeReg
from util_pred import cov_mat, fit_poly_dc

SEED      = 42
np.random.seed(SEED)

M  = Sphere()
Ex = ExponentialBarycenter()

DEGREE    = 3
N_SUBJ    = 30
N_TRAIN   = 20
N_VAL     = 5
N_TEST    = 5
N_POINTS  = 45
NOISE_STD = 0.05
LON_MAX   = 0.75 * np.pi
LAT_MAX   = np.pi / 20
DIAG_LOAD = 1e-6
TEMPLATES = ['Geo', 'Poly', 'Else']

PRED_ARGS = {'n_learn': DEGREE + 1, 'n_pred': 1, 'iterative': False}

print(f'seed={SEED}  degree={DEGREE}  split={N_TRAIN}/{N_VAL}/{N_TEST}')

ModuleNotFoundError: No module named 'jax'

## Helper functions

In [4]:
def save_path(template):
    return f'synthetic_YB_{template}.npz'

def save_data(Y, B, template):
    path = save_path(template)
    np.savez(path,
             Y=np.array(Y, dtype=object),
             B=np.array(B))
    print(f'  Saved -> {path}')

def load_data(template):
    path = save_path(template)
    data = np.load(path, allow_pickle=True)
    Y = list(data['Y'])
    B = list(data['B'])
    print(f'  Loaded <- {path}')
    return Y, B

def coarse_to_fine_grid_search(Y_val, model_fn, coarse_grid, n_refine=3, factor=4.0):
    """Two-stage coarse-to-fine grid search over lambda (ridge only)."""
    all_results = {}

    # Stage 1: coarse pass
    print(f'  Coarse pass: {[f"{l:.1e}" for l in coarse_grid]}')
    for lam in coarse_grid:
        _, m = pred(Y_val, model_fn(lam), **PRED_ARGS, prnt=False)
        all_results[lam] = m['mae']
        print(f'    lambda={lam:.1e}  val MAE={m["mae"]:.4f}')

    lam_coarse_best = min(all_results, key=all_results.get)

    # Stage 2: fine pass around best coarse lambda
    log_center = np.log10(lam_coarse_best)
    log_range  = np.log10(factor)
    fine_grid  = np.logspace(log_center - log_range, log_center + log_range, n_refine).tolist()
    fine_grid  = [l for l in fine_grid
                  if not any(abs(l - ev) / ev < 0.01 for ev in all_results)]

    if fine_grid:
        print(f'  Fine pass around lambda={lam_coarse_best:.1e}: {[f"{l:.2e}" for l in fine_grid]}')
        for lam in fine_grid:
            _, m = pred(Y_val, model_fn(lam), **PRED_ARGS, prnt=False)
            all_results[lam] = m['mae']
            print(f'    lambda={lam:.2e}  val MAE={m["mae"]:.4f}')

    return min(all_results, key=all_results.get), all_results

def run_experiment(template, Y_train, Y_val, Y_test, B_train):
    print(f"\n{'='*60}")
    print(f'Experiment: {template}')
    print(f"{'='*60}")

    n_cp = np.shape(B_train[0])[0]
    dim  = 3
    P    = PowerManifold(M, n_cp)

    # covariance
    mean_b  = Ex.compute(P, B_train, max_iter=30)
    cov_b   = cov_mat(P.metric.log, B_train, mean_b) + DIAG_LOAD * np.eye(n_cp * dim)
    eigvals = np.sort(lg.eigvalsh(cov_b))[::-1]
    print(f'  Eigenvalues: {np.round(eigvals, 6)}')

    # model factory
    def model_fn(lam):
        if lam == 0:
            return Reg(M, lag=True, degree=DEGREE)
        return RidgeReg(M, mean_b, cov_b, lam, lag=True, degree=DEGREE)

    # OLS on val and test
    _, m_ols_val = pred(Y_val,  model_fn(0), **PRED_ARGS, prnt=False)
    _, m_ols     = pred(Y_test, model_fn(0), **PRED_ARGS, prnt=False)
    mae_ols      = np.array([np.mean(m) for m in m_ols['MAE']])
    print(f'  OLS  MAE: {np.mean(mae_ols):.4f} +/- {np.std(mae_ols):.4f}')

    # coarse-to-fine grid search (ridge only)
    COARSE_GRID = [1e-5, 1e-3, 1e-1, 0.4]
    lam_star, val_results = coarse_to_fine_grid_search(
        Y_val, model_fn, COARSE_GRID, n_refine=3, factor=4.0
    )
    # compare best ridge against OLS on val
    if m_ols_val['mae'] <= val_results[lam_star]:
        lam_star = 0
    print(f'  lambda* = {lam_star}')

    # ridge on test
    _, m_ridge  = pred(Y_test, model_fn(lam_star), **PRED_ARGS, prnt=False)
    mae_ridge   = np.array([np.mean(m) for m in m_ridge['MAE']])
    improvement = 100 * (np.mean(mae_ols) - np.mean(mae_ridge)) / np.mean(mae_ols)
    print(f'  Ridge MAE: {np.mean(mae_ridge):.4f} +/- {np.std(mae_ridge):.4f}  (improvement: {improvement:.1f}%)')

    return {
        'ols_mean':    np.mean(mae_ols),
        'ols_std':     np.std(mae_ols),
        'ridge_mean':  np.mean(mae_ridge),
        'ridge_std':   np.std(mae_ridge),
        'lam_star':    lam_star,
        'improvement': improvement,
    }

## Generate Data

Generates fresh data — **does not save to disk**.  
Inspect results first, then run the **Save** cell if satisfied.

In [5]:
all_data = {}
for template in TEMPLATES:
    print(f'Generating {N_SUBJ} trajectories (template={template}) ...')
    Y, _ = sph_correlated_trjs(
        LON_MAX, LAT_MAX,
        n_trj=N_SUBJ, n_points=N_POINTS,
        noise_std=NOISE_STD, mean_curve=template
    )
    print(f'  Fitting Bezier polynomials ...')
    B, _ = fit_poly_dc(M, Y, deg=DEGREE)
    all_data[template] = {'Y': Y, 'B': B}
    print(f'  Done.')
print("Data ready. To save run 'Save' cell.")

Generating 30 trajectories (template=Geo) ...
  Fitting Bezier polynomials ...
  Done.
Generating 30 trajectories (template=Poly) ...
  Fitting Bezier polynomials ...
  Done.
Generating 30 trajectories (template=Else) ...
  Fitting Bezier polynomials ...
  Done.
Data ready (not saved). Run Save cell when satisfied.


## Load Data

Run this cell **instead of Generate** to reload previously saved data.

In [ ]:
all_data = {}
for template in TEMPLATES:
    Y, B = load_data(template)
    all_data[template] = {'Y': Y, 'B': B}

## Run Experiments

In [6]:
results = {}
for template in TEMPLATES:
    d = all_data[template]
    Y, B = d['Y'], d['B']
    results[template] = run_experiment(
        template,
        Y[:N_TRAIN],
        Y[N_TRAIN:N_TRAIN + N_VAL],
        Y[N_TRAIN + N_VAL:],
        B[:N_TRAIN]
    )


Experiment: Geo
  Eigenvalues: [1.3527e-02 7.7230e-03 4.8090e-03 4.1670e-03 3.2950e-03 2.5280e-03
 2.0310e-03 9.6900e-04 8.7200e-04 8.2700e-04 3.1300e-04 2.3600e-04
 1.0000e-06 1.0000e-06 1.0000e-06 1.0000e-06 1.0000e-06 1.0000e-06]
  OLS  MAE: 0.0288 +/- 0.0014
  Coarse pass: ['1.0e-05', '1.0e-03', '1.0e-01', '4.0e-01']
    lambda=1.0e-05  val MAE=0.0299
    lambda=1.0e-03  val MAE=0.0299
    lambda=1.0e-01  val MAE=0.0339


Exception ignored in: <function _xla_gc_callback at 0x7fd1977236d0>
Traceback (most recent call last):
  File "/srv/public/bzfnavay/venvs/.conda/envs/conda_venv/lib/python3.10/site-packages/jax/_src/lib/__init__.py", line 97, in _xla_gc_callback
    def _xla_gc_callback(*args):
KeyboardInterrupt: 


KeyboardInterrupt: 

## Table 1

In [ ]:
print(f"{'='*70}")
print('Table 1 -- MAE (radians), synthetic spherical trajectory forecasting')
print(f"{'='*70}")
print(f"{'Experiment':<12} {'OLS':<26} {'Ridge (proposed)':<26} {'Impr.':>6}  lambda*")
print('-'*70)
for name, r in results.items():
    print(f"{name:<12} "
          f"{r['ols_mean']:.4f} +/- {r['ols_std']:.4f}      "
          f"{r['ridge_mean']:.4f} +/- {r['ridge_std']:.4f}    "
          f"{r['improvement']:+.1f}%   {r['lam_star']}")
print('-'*70)
avg_imp = np.mean([r['improvement'] for r in results.values()])
print(f"{'Average improvement:':<40} {avg_imp:+.1f}%")
print(f"{'='*70}")
print(f'seed={SEED}  |  degree={DEGREE}  |  split={N_TRAIN}/{N_VAL}/{N_TEST}')
print(f'covariance: sample + {DIAG_LOAD}*I (diagonal loading)')

## Save Data

Run this cell **only when satisfied** with the results.

In [ ]:
for template in TEMPLATES:
    d = all_data[template]
    save_data(d['Y'], d['B'], template)
print('All data saved.')